段落章節

ㄧ、肺癌死亡率連續三年位居國人死亡率死因榜首

二、肺癌死亡率過去相對較不嚴重，此問題是受到近年來的影響所致。對於個人退休生活而言，及早發現與治療是重要的。對此，公共衛生政策也積極輔導。然而，公共衛生政策施政上所需要的除了人數以外，疾病盛行地區的判斷也是重要的規劃數據。但相較於全因死亡率資料完整，個別死因死亡率因為較為稀疏，所以在估計變異收斂速度較慢。

三、針對此問題，人口統計上常見的方法之一為合併較大樣本地區形成參考母體。在參考母體選擇的方向上，「陳玟儒（2024），台灣居民的全死因與」一文中透過探索性分析以及分群分析的方法達到良好的估計效及主要死因時空分析於上述觀點，我們這次的期末報告也採用兩階段的研究策略。首先，我們使用主成分分析肺癌的標準化發生率。其次，我們透過不同分群分析的方法，合併相似地區作作為群體。藉此專案分析，我們希望能夠提供未來建立公共衛生上疾病盛行率以及建立小地區疾病盛行率。

由於本研究各年度肺癌年齡標準化發生率具有相同量綱（每十萬人口發生率），理論上可直接使用 covariance matrix 進行 PCA。然而，各年份之縣市間變異程度仍存在差異，特別是近年資料呈現較高之區域異質性，因此若直接使用 covariance matrix，可能使高變異年份對主成分方向產生較大影響。

基於本研究目的著重於**不同縣市肺癌發生趨勢之相對型態與相似性**，而非單一年度變異程度之大小，因此本研究採用 correlation matrix 進行主成分分析，相當於先對各年份資料進行標準化後再進行 PCA。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd

# Adjust the path to your CSV file in Google Drive
csv_file_path = '/content/drive/MyDrive/多變量分析/肺癌罹病率資料.csv'

try:
    df = pd.read_csv(csv_file_path)
    print("CSV file loaded successfully!")
    display(df.head())
except FileNotFoundError:
    print(f"Error: The file '{csv_file_path}' was not found. Please check the path.")
except Exception as e:
    print(f"An error occurred: {e}")

In [ ]:
df_filtered = df[(df['性別'] == '全') &(df['縣市別'] != '全國') & (df['癌症別'] == '肺、支氣管及氣管')]
display(df_filtered)

In [ ]:
pivot_table_age_std = df_filtered.pivot_table(
    index='縣市別',
    columns='癌症診斷年',
    values='年齡標準化發生率  WHO 2000世界標準人口 (每10萬人口)',
    aggfunc='mean' # Assuming you want the mean if there are multiple entries
)
display(pivot_table_age_std)
pivot_table_age_std_country = df_filtered.pivot_table(
    index="癌症診斷年",
    columns="縣市別",
    values="年齡標準化發生率  WHO 2000世界標準人口 (每10萬人口)",
    aggfunc="mean"
)

In [ ]:
correlation_matrix = pivot_table_age_std_country.corr()
correlation_matrix_round4 = correlation_matrix.round(3)
display((correlation_matrix_round4.head()))

下面這段Code用於安裝Global的中文繪圖

In [ ]:
import matplotlib.font_manager as fm
# 如果是COLAB要先下載繁體中文字型
!wget -O TaipeiSansTCBeta-Regular.ttf https://drive.google.com/uc?id=1eGAsTN1HBpJAkeVM57_C7ccp7hbgSz3_&export=download
#將字型加入，確認系統字型
fm.fontManager.addfont('TaipeiSansTCBeta-Regular.ttf')
#列出所有字體
[f.name for f in fm.fontManager.ttflist]
#全局參數設定 runtime configuration，字體設為可呈現中文的字體
plt.rc('font', family='Taipei Sans TC Beta')

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm


plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.show()

In [ ]:
cov_matrix = pivot_table_age_std_country.cov()
cov_round4= cov_matrix.round(4)
display((cov_round4.head()))

In [ ]:
plt.figure(figsize=(10,8))
sns.heatmap(cov_matrix, annot=True, cmap="coolwarm", fmt=".2f")
plt.show()


Finally, let's visualize the reordered correlation matrix using a heatmap.

In [ ]:
plt.figure(figsize=(12, 10))
sns.heatmap(reordered_correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5)
plt.title('Correlation Matrix of Age-Standardized Incidence Rates (Ordered by Hclust)')
plt.show()

共變異數矩陣版本的PCA函數使用的是sklearn.decomposition函數包的PCA

In [ ]:
from sklearn.decomposition import PCA
import numpy as np

# Initialize PCA. We will keep all components for the scree plot first.
pca = PCA()
pca.fit(correlation_matrix)

# Explained variance ratio
explained_variance_ratio = pca.explained_variance_ratio_

# Cumulative explained variance
cumulative_explained_variance = np.cumsum(explained_variance_ratio)

print("Explained Variance Ratio:")
print(explained_variance_ratio)
print("\nCumulative Explained Variance:")
print(cumulative_explained_variance)

若要對相關係數矩陣計算PCA，則要匯入StandardScaler以及sklearn.decompsition中的PCA。
先用scaler.fit_transform將共變異數矩陣中心化

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)   # 标准化
pca = PCA()
pca.fit(X_scaled)

或者是可以用基礎版本的模式，即以特徵值的方式計算

## Principal Component Analysis (PCA) on Correlation Matrix

Now, let's perform PCA on the `correlation_matrix` to reduce dimensionality and identify underlying patterns. We'll start by importing `PCA` from `sklearn.decomposition` and fitting it to our data.

In [ ]:
import scipy.cluster.hierarchy as sch
import matplotlib.pyplot as plt
import seaborn as sns

# Perform hierarchical clustering
linked = sch.linkage(correlation_matrix, method='ward')

# Get the order of the clusters
cluster_order = sch.dendrogram(linked, no_plot=True)['leaves']

# Reorder the correlation matrix based on the clustering result
reordered_correlation_matrix = correlation_matrix.iloc[cluster_order, cluster_order]
display(reordered_correlation_matrix.head())

In [ ]:
from sklearn.decomposition import PCA
import numpy as np

# Initialize PCA. We will keep all components for the scree plot first.
pca = PCA()
pca.fit(correlation_matrix)

# Explained variance ratio
explained_variance_ratio = pca.explained_variance_ratio_

# Cumulative explained variance
cumulative_explained_variance = np.cumsum(explained_variance_ratio)

print("Explained Variance Ratio:")
print(explained_variance_ratio)
print("\nCumulative Explained Variance:")
print(cumulative_explained_variance)

### Scree Plot

A scree plot helps visualize the eigenvalues (explained variance) for each principal component, allowing us to determine the optimal number of components to retain.

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(explained_variance_ratio) + 1), explained_variance_ratio, marker='o', linestyle='--')
plt.title('Scree Plot')
plt.xlabel('Number of Components')
plt.ylabel('Explained Variance Ratio')
plt.grid(True)
plt.show()

### Cumulative Explained Variance Plot

This plot shows the cumulative sum of explained variance, which helps in deciding how many principal components are needed to explain a certain percentage of the total variance.

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(cumulative_explained_variance) + 1), cumulative_explained_variance, marker='o', linestyle='--')
plt.axhline(y=0.8, color='r', linestyle='-', label='80% Cumulative Variance')
plt.title('Cumulative Explained Variance Plot')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance Ratio')
plt.grid(True)
plt.legend()
plt.show()

### Biplot

A biplot visualizes the relationship between the observations (rows) and the variables (columns) in the PCA space. It helps in understanding how original variables contribute to the principal components and how observations are grouped.

In [ ]:
# For biplot, we usually consider the first two principal components
pca_2_components = PCA(n_components=2)
components = pca_2_components.fit_transform(correlation_matrix)
loadings = pca_2_components.components_.T * np.sqrt(pca_2_components.explained_variance_)

fig, ax = plt.subplots(figsize=(12, 12))
ax.scatter(components[:, 0], components[:, 1], alpha=0.5)

for i, feature in enumerate(correlation_matrix.columns):
    ax.arrow(0, 0, loadings[i, 0], loadings[i, 1], color='r', alpha=0.8)
    ax.text(loadings[i, 0] * 1.15, loadings[i, 1] * 1.15, feature, color='g', ha='center', va='center')

ax.set_xlabel('Principal Component 1')
ax.set_ylabel('Principal Component 2')
ax.set_title('Biplot of PCA on Correlation Matrix')
ax.grid(True)
plt.show()

## Cluster Analysis

We will now perform cluster analysis on the `pivot_table_age_std` to group similar `縣市別` (cities/counties) based on their age-standardized incidence rates over the years. We'll test different distance measures and clustering algorithms as requested.

### Data Preparation for Clustering

First, we need to handle any missing values in `pivot_table_age_std` and scale the data. Clustering algorithms are sensitive to missing values and the scale of features.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from scipy.spatial.distance import pdist, squareform
from scipy.cluster.hierarchy import linkage, dendrogram
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Ensure the data is numeric and handle potential NaNs
# For clustering, we need to ensure no NaNs. Let's impute with the mean of each column (year).
clustering_data = pivot_table_age_std.copy()

# Impute NaNs with the mean of each column
for col in clustering_data.columns:
    if clustering_data[col].isnull().any():
        clustering_data[col] = clustering_data[col].fillna(clustering_data[col].mean())

# Standardize the data. We want to cluster the rows (縣市別).
# So we scale features (years) for each observation (縣市別).
scaler = StandardScaler()
scaled_clustering_data = scaler.fit_transform(clustering_data)

# Convert back to DataFrame for easier inspection, keeping original index and columns
scaled_clustering_df = pd.DataFrame(scaled_clustering_data, index=clustering_data.index, columns=clustering_data.columns)
display(scaled_clustering_df.head())


### K-Means Clustering

We'll apply K-Means clustering. For demonstration, we'll choose a number of clusters (e.g., `k=4`). To visualize the clusters, we'll reduce the dimensionality using PCA to two components.

In [ ]:
from sklearn.decomposition import PCA

k = 4  # Let's start with 4 clusters for demonstration
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10) # n_init is set to 'auto' or explicit int in newer versions
kmeans_labels = kmeans.fit_predict(scaled_clustering_data)

# Add cluster labels to the original DataFrame
clustering_data['KMeans_Cluster'] = kmeans_labels
display(clustering_data.head())

# Visualize K-Means clusters using PCA for dimensionality reduction
pca_kmeans = PCA(n_components=2)
components_kmeans = pca_kmeans.fit_transform(scaled_clustering_data)

plt.figure(figsize=(10, 8))
sns.scatterplot(
    x=components_kmeans[:, 0],
    y=components_kmeans[:, 1],
    hue=kmeans_labels,
    palette='viridis',
    s=100,
    alpha=0.8
)
plt.title(f'K-Means Clustering (k={k}) - PCA Reduced')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.grid(True)
plt.show()

### Hierarchical Clustering

Now, let's explore hierarchical clustering using different distance metrics and linkage methods. We'll use Pearson Correlation and Spearman Correlation as distance measures, and 'single', 'complete', and 'average' as linkage methods. For each combination, we'll display a dendrogram.

#### Hierarchical Clustering with Pearson Correlation Distance

In [ ]:
# Calculate Pearson correlation-based distance (1 - correlation)
# pdist computes a condensed distance matrix
pearson_distances = pdist(scaled_clustering_data, metric='correlation')

# Linkage methods
linkage_methods = ['single', 'complete', 'average']

plt.figure(figsize=(18, 12))

for i, method in enumerate(linkage_methods):
    plt.subplot(3, 1, i + 1)
    linked = linkage(pearson_distances, method=method)
    dendrogram(linked,
               orientation='top',
               labels=clustering_data.index.tolist(),
               distance_sort='descending',
               show_leaf_counts=True,
               color_threshold=0.7 * max(linked[:,2])) # Adjust threshold for better visibility
    plt.title(f'Hierarchical Clustering (Pearson - {method} Linkage)')
    plt.xlabel('縣市別')
    plt.ylabel('Distance')

plt.tight_layout()
plt.show()

#### Hierarchical Clustering with Spearman Correlation Distance

In [ ]:
from scipy.stats import rankdata

# Define a custom function for Spearman correlation distance
def spearman_correlation_distance(u, v):
    # Calculate ranks for each vector u and v
    rank_u = rankdata(u)
    rank_v = rankdata(v)
    # Calculate Pearson correlation coefficient between the ranks
    corr_matrix = np.corrcoef(rank_u, rank_v)
    corr = corr_matrix[0, 1]

    # Handle NaN correlation (e.g., if one or both vectors are constant)
    if np.isnan(corr):
        # A common practice is to assign a neutral distance of 1.0 (equivalent to correlation 0)
        return 1.0
    return 1 - corr

# Calculate Spearman correlation-based distance using the custom metric
spearman_distances_condensed = pdist(scaled_clustering_data, metric=spearman_correlation_distance)

plt.figure(figsize=(18, 12))

for i, method in enumerate(linkage_methods):
    plt.subplot(3, 1, i + 1)
    linked = linkage(spearman_distances_condensed, method=method)
    dendrogram(linked,
               orientation='top',
               labels=clustering_data.index.tolist(),
               distance_sort='descending',
               show_leaf_counts=True,
               color_threshold=0.7 * max(linked[:,2])) # Adjust threshold for better visibility
    plt.title(f'Hierarchical Clustering (Spearman - {method} Linkage)')
    plt.xlabel('縣市別')
    plt.ylabel('Distance')

plt.tight_layout()
plt.show()